# Projekt: Jobbannons-filter
**Namn:** Elias Paleovrachas Haag
**Kurs:** Utveckling med Python, grund
**Verktyg:** VS Code, Jobtech API, Git/GitHub

### Projektbeskrivning
Detta projekt bygger en automatiserad datapipeline som hämtar realtidsdata om jobbannonser för programmeringsspråk från Arbetsförmedlingens **JobTech API**. Datamängden valideras och tvättas med objektorienterade principer (OOP) och sparas säkert som strukturerad rådata (CSV) samt renad data med GDPR-metadata (JSON).

## 1. Klasser och valideringslogik (OOP)
I denna cell definieras inställningslistor för godkända språk och orter samt projektets klassstruktur:
* **`Jobb_annons` (Basklass/Förälderklass):** Håller grundläggande attribut (`language`, `location`, `date_created`) samt valideringsmetoderna `validate_language()` och `validate_location()`.
* **`Distans_jobb` (Barnklass/Arv):** Ärver alla attribut och metoder från förälderklassen via `super().__init__()` och utökas med det unika attributet `distans`.

In [1]:
import csv
import os
from datetime import datetime
import requests
import json



accepted_languages = ["Python", "Java", "C++", "C#", "SQL"]
accepted_locations = [
    "Stockholm", "Göteborg", "Malmö", "Uppsala", "Västerås", 
    "Örebro", "Linköping", "Helsingborg", "Jönköping", "Norrköping",
    "Umeå", "Lund", "Borås", "Gävle", "Eskilstuna", "Karlstad, Solna"
]

class Jobb_annons:
    def __init__(self, language, location, date_created):
        self.language = language
        self.location = location
        self.date_created = date_created
        
#Koppling till yrkesrollen: Precis som Lovable gör, hjälper denna funktion att ta emot de värde som användaren ger och sedan gå igenom den godkända listan för att säkerställa att inga felaktiga värden når AI-modellen.
    def validate_language(self):
      return self.language in accepted_languages
        
    def validate_location(self):
      if not self.location or self.location in ["None", "Okänd ort"]:
         return False
      return True

class Distans_jobb(Jobb_annons):
   def __init__(self, language, location, distans, date_created):
      super().__init__(language, location, date_created)
      self.distans = distans      





## 2. Datainsamling via JobTech API & Rådatalagring (CSV)
Denna cell hanterar den automatiska datainsamlingen från omvärlden:
1. Mappstrukturen (`data/raw` och `output`) skapas automatiskt med `os.makedirs()`.
2. En `for`-loop går igenom varje godkänt språk och skickar ett HTTP-anrop till JobTech API via biblioteket `requests`.
3. API-svaret (JSON) omvandlas till `Distans_jobb`-objekt med tidsstämpel och distans-status.
4. Rådatan sparas till filen `data/raw/raw_data.csv` för framtida spårbarhet.


In [2]:
os.makedirs(os.path.join("data", "raw"), exist_ok=True)
os.makedirs("output", exist_ok=True)

riktiga_annonser_lista = []
tidsstampel = datetime.now().strftime("%Y-%m-%d %H:%M")

print("Anropar JobTech API för alla språk...")

for sprak in accepted_languages:
    url = f"https://jobsearch.api.jobtechdev.se/search?q={sprak}&limit=5"
    response = requests.get(url)

    response = requests.get(url)

    
    if response.status_code == 200:
        api_data = response.json()
        annonser = api_data.get("hits", [])

        for annons in annonser:
            ort = annons.get("workplace_address", {}).get("municipality", "Okänd ort")
            is_remote = annons.get("workplace_address", {}).get("workplace_remote_approved", False)

            ny_annons_objekt = Distans_jobb(
                language=sprak,
                location=ort,
                distans=is_remote,
                date_created=tidsstampel,
            )

            riktiga_annonser_lista.append(ny_annons_objekt)

        print(f"Hämtade {len(annonser)} st jobb för {sprak}")

    else:
        print(f"Kunde inte hämta data för {sprak}. Felkod: {response.status_code}")

  
csv_file_path = os.path.join("data", "raw", "raw_data.csv")

with open(csv_file_path, "w", encoding="utf-8") as min_fil:
    min_fil.write("Language,Location,Distans,Date_Created\n")
    for jobb in riktiga_annonser_lista:
        min_fil.write(
            f"{jobb.language},{jobb.location},{jobb.distans},{jobb.date_created}\n"
        )


print(f"Framgång! Riktig data har sparats i: {csv_file_path}")



Anropar JobTech API för alla språk...
Hämtade 5 st jobb för Python
Hämtade 5 st jobb för Java
Hämtade 5 st jobb för C++
Hämtade 10 st jobb för C#
Hämtade 5 st jobb för SQL
Framgång! Riktig data har sparats i: data\raw\raw_data.csv


## 3. Datatvätt (Data Cleaning), felhantering och JSON-export
Här byggs "filtret" som säkerställer att felaktig rådata inte når en framtida AI-modell:
1. Rådatan läses in från CSV-filen inuti ett `try/except`-block för säker filhantering mot t.ex. `FileNotFoundError`.
2. Varje rad tvättas och valideras med klassens objektmetoder. Felaktiga eller ogiltiga värden sorteras bort och en rapport skrivs ut i terminalen.


Jag använde mig av CSV som modul då de jobbannonser jag arbetar med gör det enklast att spara de i en tabell format för att göra det lätt läst och kunna dela vidare detta.

In [3]:

clean_data_list = []
kasserade_rader = 0

try:
    csv_file_path = os.path.join("data", "raw", "raw_data.csv")
    with open(csv_file_path, "r", encoding="utf-8") as fil:
        rader = fil.readlines()

        for rad in rader[1:]:
            rad = rad.strip()
            if not rad:
                continue

            delar = rad.split(",")
            if len(delar) < 4:
                kasserade_rader += 1
                continue

            sprak = delar[0]
            ort = delar[1]
            distans = delar[2] == "True"
            skapad_tid = delar[3]

            test_objekt = Distans_jobb(
                language=sprak,
                location=ort,
                distans=distans,
                date_created=skapad_tid,
            )

            if (test_objekt.validate_language() and test_objekt.validate_location()):
                clean_data_list.append({
                    "language": test_objekt.language,
                    "location": test_objekt.location.strip().title(),
                    "is_remote": test_objekt.distans,
                    "date_created": test_objekt.date_created,
                })            
            else:
                kasserade_rader += 1
                print(f"Kasserade: Språk ='{test_objekt.language}',"
                      f"Ort='{test_objekt.location}'")

        print("--- Rapport från Datatvätten ---")
        print(f"Antal godkända rader sparade: {len(clean_data_list)}")
        print(f"Antal rader kasserade p.g.a felaktiga värden: {kasserade_rader}")
        print("--------------------------------")

except FileNotFoundError:
    print(f"Fel: Kunde inte hitta rådata-filen. Kör Cell 2 först!")

if clean_data_list:
    json_output = {
        "source": "Jobtech API (Arbetsförmedlingen)",
        "gdpr_compliant": True,
        "created": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "data": clean_data_list,
    }

    json_file_path = os.path.join("output", "clean_data.json")

    try:
        with open(json_file_path, "w", encoding="utf-8") as json_fil:
            json.dump(json_output, json_fil, ensure_ascii=False, indent=4)
        print(f"Framgång! Din renade data har sparats i: {json_file_path}")
    except Exception as e:
        print(f"Kunde inte spara JSON-filen: {e}")

        

Kasserade: Språk ='Python',Ort='None'
Kasserade: Språk ='Python',Ort='None'
--- Rapport från Datatvätten ---
Antal godkända rader sparade: 28
Antal rader kasserade p.g.a felaktiga värden: 2
--------------------------------
Framgång! Din renade data har sparats i: output\clean_data.json


## 4. Etik, Riskanalys & Framtidsarkitektur

### Riskanalys enligt EU AI Act
* **Riskkategori:** Projektet klassificeras som **Minimal risk** då det är ett analysverktyg för offentliga jobbannonser och inte utför automatisk rekrytering eller profilering av enskilda personer.
* **Datakvalitet:** Datatvätten rensar bort korrupta orter och saknade mätvärden, vilket minskar risken för att en framtida AI-modell drar felaktiga slutsatser om arbetsmarknaden.

### Reflektion: Multi-Agent System (MAS)
Om detta projekt skulle utvecklas till ett Multi-Agent System (MAS) istället för ett linjärt program, skulle ansvaret delas upp på två specialiserade agenter:
1. **Data-agenten (Jobb-hämtare):** Har som enda uppgift att kommunicera med JobTech API, hantera rate limits och spara rådata till CSV.
2. **Analys-agenten (Datatvättare):** Övervakar när ny rådata dyker upp, validerar objekten, rensar bort skräpdata och exporterar den färdiga JSON-filen med GDPR-metadata.
